UNCOMPRESSING THE FILE INTO A TEXT FILE SO ITS EASIER TO LOAD

In [ ]:
# import gzip
# import shutil
# from pathlib import Path

# DATA_PATH = Path("data_windows")

# gz_path = DATA_PATH / "auth.txt.gz"
# out_path = DATA_PATH / "auth.txt"

# with gzip.open(gz_path, "rb") as f_in:
#     with open(out_path, "wb") as f_out:
#         shutil.copyfileobj(f_in, f_out)

# print("Decompression complete.")


Decompression complete.


FILTERING THE REDTEAM ROWS AND BUILDING A PARQUET FILE FOR EASIER ACCESS TO DATASET

In [10]:
import pandas as pd
redteam_df = pd.read_csv(
    DATA_PATH / "redteam.txt.gz",
    compression="gzip",
    header=None
)

redteam_df.columns = [
    "timestamp",
    "user",
    "source_computer",
    "destination_computer"
]

redteam_df["timestamp"] = redteam_df["timestamp"].astype("int32")



WINDOW OF ATTACKS

In [12]:
redteam_df = pd.read_csv(
    DATA_PATH / "redteam.txt.gz",
    compression="gzip",
    header=None
)

redteam_df.columns = [
    "timestamp",
    "user",
    "source_computer",
    "destination_computer"
]

redteam_df["timestamp"] = redteam_df["timestamp"].astype("int32")

rt_min = redteam_df["timestamp"].min()
rt_max = redteam_df["timestamp"].max()

print("Redteam window:", rt_min, "to", rt_max)


Redteam window: 150885 to 2557047


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_PATH = Path("data_windows")
BUFFER = 2000
start_time = rt_min - BUFFER
end_time = rt_max + BUFFER

chunk_size = 1_000_000
output_file = "auth_time_window.parquet"

first = True
collected = 0
max_rows = None   # safe limit

for chunk in pd.read_csv(
    DATA_PATH / "auth.txt",
    header=None,
    usecols=[0,1,3,4,8],
    chunksize=chunk_size
):

    chunk.columns = [
        "timestamp",
        "source_user",
        "source_computer",
        "destination_computer",
        "success"
    ]

    chunk["timestamp"] = chunk["timestamp"].astype("int32")

    filtered = chunk[
        (chunk["timestamp"] >= start_time) &
        (chunk["timestamp"] <= end_time)
    ]

    if len(filtered) > 0:
        if first:
            filtered.to_parquet(output_file, engine="fastparquet")
            first = False
        else:
            filtered.to_parquet(output_file, engine="fastparquet", append=True)

        collected += len(filtered)



print("Saved rows:", collected)


KeyboardInterrupt: 

In [14]:
auth_df = pd.read_parquet("auth_time_window.parquet", engine="fastparquet")

auth_df = auth_df.merge(
    redteam_df,
    how="left",
    left_on=["timestamp", "source_user", "source_computer", "destination_computer"],
    right_on=["timestamp", "user", "source_computer", "destination_computer"]
)

auth_df["label"] = auth_df["user"].notna().astype("int8")
auth_df.drop(columns=["user"], inplace=True)

print(auth_df["label"].value_counts())


label
0    3762195
1         10
Name: count, dtype: int64
